In [0]:
from pyspark.sql.functions import col, sum, round

# 1. Lemos as tabelas perfeitas que você preparou na Silver
df_pizzas = spark.read.table("tabela_silver_vendas_pizzas")
df_caixa = spark.read.table("tabela_silver_caixa")

# 2. A MÁGICA DO GROUP BY: Somamos o dinheiro por dia ANTES de juntar
df_caixa_diario = (
    df_caixa
    .groupBy("data")
    .agg(round(sum("valor_receita"), 2).alias("faturamento_diario"))
)

# 3. O JOIN SEGURO (1 linha de pizza para 1 linha de faturamento)
df_gold_vendas = df_pizzas.join(df_caixa_diario, on="data", how="inner")

# 4. Ordenamos pela data para o dashboard contar a história na ordem certa
df_gold_vendas = df_gold_vendas.orderBy("data")

print("--- RESULTADO GOLD: VENDAS E FATURAMENTO DIÁRIO ---")
display(df_gold_vendas)

In [0]:
from pyspark.sql.functions import to_date, concat, lit

# 1. Transformamos o texto "DD/MM" numa data real ("DD/MM/YYYY") 
# Concatenamos o ano de 2026 (ou o ano correspondente aos dados) ao final da string
df_gold_vendas = df_gold_vendas.withColumn(
    "data_real", 
    to_date(concat(col("data"), lit("/2026")), "dd/MM/yyyy")
)

# 2. Agora o Spark vai ordenar de forma cronológica (Janeiro, Fevereiro, Março...) em vez de alfabética
df_gold_vendas = df_gold_vendas.orderBy("data_real")

# 3. Guardamos as colunas finais
df_gold_vendas = df_gold_vendas.select("data_real", "qtd_pizzas", "faturamento_diario")

display(df_gold_vendas)

In [0]:
# Salvando tabela gold vendas pizzaria em formato delta

df_gold_vendas.write.mode("overwrite").format("delta").saveAsTable("tabela_gold_vendas_pizzaria")